In [32]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio
from IPython.display import display, Markdown
from pydantic import BaseModel
from datetime import datetime
import json
from mcp.server.fastmcp import FastMCP
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio
from IPython.display import display, Markdown
import yfinance as yf
load_dotenv(override=True)


True

In [33]:
from pydantic import BaseModel
from datetime import datetime
import json
import yfinance as yf

INITIAL_BALANCE = 10_000.0
SPREAD = 0.002

_database = {}

def get_share_price(symbol: str) -> float:
    try:
        ticker = yf.Ticker(symbol)
        price = ticker.history(period="1d")["Close"].iloc[-1]
        return float(price)
    except:
        return 0.0

def write_account(name: str, data: dict):
    _database[name.lower()] = data

def read_account(name: str):
    return _database.get(name.lower())

def write_log(name: str, type_: str, message: str):
    print(f"[{name}] {type_}: {message}")

class Transaction(BaseModel):
    symbol: str
    quantity: int
    price: float
    timestamp: str
    rationale: str

    def total(self) -> float:
        return self.quantity * self.price

class Account(BaseModel):
    name: str
    balance: float = INITIAL_BALANCE
    strategy: str = ""
    holdings: dict[str, int] = {}
    transactions: list[Transaction] = []
    portfolio_value_time_series: list[tuple[str, float]] = []

    @classmethod
    def get(cls, name: str):
        fields = read_account(name.lower())
        if not fields:
            fields = {
                "name": name.lower(),
                "balance": INITIAL_BALANCE,
                "strategy": "",
                "holdings": {},
                "transactions": [],
                "portfolio_value_time_series": []
            }
            write_account(name.lower(), fields)
        return cls(**fields)

    def save(self):
        write_account(self.name.lower(), self.model_dump())

    def buy_shares(self, symbol: str, quantity: int, rationale: str) -> str:
        price = get_share_price(symbol)
        if price == 0:
            raise ValueError(f"Unrecognized symbol {symbol}")
        buy_price = price * (1 + SPREAD)
        total_cost = buy_price * quantity
        if total_cost > self.balance:
            raise ValueError("Insufficient funds")
        self.holdings[symbol] = self.holdings.get(symbol, 0) + quantity
        self.balance -= total_cost
        ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        trx = Transaction(symbol=symbol, quantity=quantity, price=buy_price, timestamp=ts, rationale=rationale)
        self.transactions.append(trx)
        self.save()
        write_log(self.name, "account", f"Bought {quantity} {symbol}")
        return "Buy completed.\n" + self.report()

    def sell_shares(self, symbol: str, quantity: int, rationale: str) -> str:
        if self.holdings.get(symbol, 0) < quantity:
            raise ValueError("Not enough shares")
        price = get_share_price(symbol)
        sell_price = price * (1 - SPREAD)
        proceeds = sell_price * quantity
        self.holdings[symbol] -= quantity
        if self.holdings[symbol] == 0:
            del self.holdings[symbol]
        self.balance += proceeds
        ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        trx = Transaction(symbol=symbol, quantity=-quantity, price=sell_price, timestamp=ts, rationale=rationale)
        self.transactions.append(trx)
        self.save()
        write_log(self.name, "account", f"Sold {quantity} {symbol}")
        return "Sell completed.\n" + self.report()

    def calculate_portfolio_value(self) -> float:
        total = self.balance
        for sym, qty in self.holdings.items():
            total += get_share_price(sym) * qty
        return total

    def report(self) -> str:
        pv = self.calculate_portfolio_value()
        ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        self.portfolio_value_time_series.append((ts, pv))
        self.save()
        data = self.model_dump()
        data["total_portfolio_value"] = pv
        return json.dumps(data, indent=2)

In [34]:
account = Account.get("JARIF")
account.buy_shares("AMZN", 3, "Strategic investment in e-commerce leader with strong AWS cloud revenue diversification and consistent market dominance")
print(account.report())

[jarif] account: Bought 3 AMZN
{
  "name": "jarif",
  "balance": 9319.802323669433,
  "strategy": "",
  "holdings": {
    "AMZN": 3
  },
  "transactions": [
    {
      "symbol": "AMZN",
      "quantity": 3,
      "price": 226.73255877685546,
      "timestamp": "2025-11-25 16:57:18",
      "rationale": "Strategic investment in e-commerce leader with strong AWS cloud revenue diversification and consistent market dominance"
    }
  ],
  "portfolio_value_time_series": [
    [
      "2025-11-25 16:57:18",
      9998.642320007324
    ],
    [
      "2025-11-25 16:57:19",
      9998.642320007324
    ]
  ],
  "total_portfolio_value": 9998.642320007324
}


In [35]:
instructions = """You are a professional portfolio manager and financial advisor. 
You manage client investment accounts with expertise in:
- Portfolio analysis and risk assessment
- Strategic stock recommendations based on market trends
- Detailed transaction execution with comprehensive rationale
- Real-time portfolio performance tracking
Always provide detailed explanations and market context for your recommendations."""


request = "My name is JARIF. Please provide a comprehensive overview of my investment portfolio including current cash position, stock holdings with current market values, and overall portfolio performance."
model = "gpt-4o-mini"

params = {"command": "python", "args": ["accounts_server.py"]}

async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="account_manager", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("account_query"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"
HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1 204 No Content"
HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1 204 No Content"
HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"


### Overview of JARIF's Investment Portfolio

#### Current Cash Position
- **Cash Available**: $10,000

#### Stock Holdings
- **Current Holdings**: You currently do not have any stock holdings in your investment portfolio.

#### Overall Portfolio Performance
Given that you have $10,000 in cash and no stock investments, your portfolio is currently not exposed to market fluctuations. This means your portfolio performance is stable, but it may also miss out on potential market gains.

### Recommendations

1. **Diversification**: Consider investing in a diversified mix of stocks or ETFs to mitigate risks while aiming for returns.
2. **Market Trends**: Keep an eye on sectors that are performing well, such as technology or green energy.
3. **Assess Risk**: Based on your risk tolerance, adjust your asset allocation accordingly.
4. **Revisit Strategy**: If you have a specific investment strategy or goal, it might be worth discussing to align your cash reserves with those objectives.

If you have specific investment interests or sectors you'd like to explore, let me know, and I can provide tailored recommendations.

In [ ]:
request = """Analyze the current EV market landscape and execute a strategic purchase of 5 shares of TSLA. 
Provide rationale considering: Tesla's manufacturing capacity expansion, competitive positioning against traditional automakers, 
battery technology advantages, and potential risks from increased competition in the EV sector."""

async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="account_manager", instructions=instructions, model=model, mcp_servers=[mcp_server])
    result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1 204 No Content"
HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"
HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"
HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1 204 No Content"
HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1 204 No Content"
HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"
HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1 204 No Content"
HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"


### Transaction Summary
**Investment**: Purchased 5 shares of Tesla (TSLA)  
**Price per Share**: $418.62  
**Total Cost**: $2,093.08  
**Remaining Balance**: $7,906.92

### Rationale for Purchase
1. **Manufacturing Capacity Expansion**:
   - Tesla is actively increasing its manufacturing footprint through the establishment of new gigafactories, which are expected to ramp up production in response to the growing global demand for electric vehicles (EVs). This expansion will enable Tesla to produce more vehicles and fulfill sales while maintaining market leadership.

2. **Competitive Positioning**:
   - Tesla has successfully carved a niche in the EV market, establishing strong brand loyalty. While traditional automakers like Ford and GM are escalating their efforts in the EV sphere, they are still playing catch-up to Tesla's rapid market penetration and strong consumer recognition.

3. **Battery Technology Advantages**:
   - Tesla's investments in battery technology, especially in enhancing energy density and reducing production costs, provide a significant competitive edge. Superior battery performance translates to longer ranges and faster charging times, making Tesla vehicles more appealing in a crowded market.

4. **Potential Risks**:
   - Although there’s a rising threat from increased competition within the EV sector, Tesla has demonstrated resilience and adaptability, maintaining its market position. The company's established leadership gives it leverage to navigate challenges effectively.

### Conclusion
The decision to acquire Tesla shares aligns with a growth-oriented investment strategy aimed at capitalizing on the exciting developments in the EV market. Given Tesla's production expansion and technological advancements, this investment is expected to yield significant long-term returns despite the potential risks posed by increasing competition.

HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1 204 No Content"
